In [12]:
# --- Helper Functions ---
def Fetch_Related_Drugs(Ing_lst, current_id):
    if not Ing_lst:
        return pd.DataFrame(columns=["RXCUI", "STR", "Product_Name"])

    ing_tuple = tuple(Ing_lst) if len(Ing_lst) > 1 else f"('{Ing_lst[0]}')"

    # Handle single vs multi ingredient properly
    if len(Ing_lst) == 1:
        having_clause = "COUNT(DISTINCT r1.RXCUI1) >= 1"
    else:
        having_clause = """
            COUNT(DISTINCT r1.RXCUI1) > 0
            AND COUNT(DISTINCT r1.RXCUI1) < :ing_count
        """

    query = f"""
    SELECT 
        r2.RXCUI as RXCUI, 
        r2.STR as STR
    FROM RXNREL r1
    JOIN RXNCONSO r2 ON r1.RXCUI2 = r2.RXCUI
    WHERE r1.RXCUI1 IN {ing_tuple} 
      AND r2.TTY = 'DP'
      AND r2.RXCUI != :current_id
    GROUP BY r2.RXCUI, r2.STR
    HAVING {having_clause}
    """
    
    with engine.connect() as conn:
        res = pd.read_sql(text(query), conn, params={
            'current_id': current_id, 
            'ing_count': len(Ing_lst)
        })
    
    if not res.empty:
        res = res[res['STR'].str.contains(r'\[.*\]', na=False)].copy()
        res["Product_Name"] = res["STR"].str.extract(r'\[(.*?)\]')
        res["Product_Name"] = res["Product_Name"].str.title()
        res = res.drop_duplicates(subset=["Product_Name"])
        res = res[res["Product_Name"].str.lower() != "generic"]
        res.drop_duplicates(subset="RXCUI", inplace=True)
        res.reset_index(drop=True, inplace=True)
        
    return res

def extract_name(df):
    if df.empty:
        return df
    df["Product_Name"] = df["STR"].str.extract(r'\[(.*?)\]')
    df["Product_Name"] = df["Product_Name"].fillna("Generic")
    df["Product_Name"] = df["Product_Name"].str.title()
    df = df.drop_duplicates(subset=["Product_Name"])
    return df

def Searchbar(term):
    sql = text("SELECT RXCUI, STR FROM RXNCONSO WHERE STR LIKE :term AND TTY IN ('DP')")
    with engine.connect() as conn:
        df = pd.read_sql(sql, conn, params={'term': f'%{term}%'})
    return extract_name(df)

def Fetch_Ingredients(ID):
    query = text("""
        SELECT r.RXCUI2 as Ingredient_ID, c.STR as Full_Ingredient
        FROM RXNCONSO c
        JOIN RXNREL r ON c.RXCUI = r.RXCUI2
        WHERE r.RXCUI1 = :id AND c.TTY = "SCDC"
        GROUP BY Ingredient_ID, Full_Ingredient;
    """)
    with engine.connect() as conn:
        df = pd.read_sql(query, conn, params={"id": ID})
    
    parsed_data = []
    for _, row in df.iterrows():
        match = re.search(r"(.+?)\s+(\d+(?:\.\d+)?)\s+MG", row["Full_Ingredient"], re.IGNORECASE)
        if match:
            parsed_data.append({
                "Ingredient_ID": row["Ingredient_ID"],
                "Ingredient": match.group(1).strip(),
                "Concentration": float(match.group(2)) 
            })
        else:
            parsed_data.append({
                "Ingredient_ID": row["Ingredient_ID"],
                "Ingredient": row["Full_Ingredient"],
                "Concentration": 0.0 
            })
    return pd.DataFrame(parsed_data)

def Fetch_Exact_Drugs(Ing_lst, ing_names, current_id):
    if not Ing_lst:
        return pd.DataFrame()

    ing_tuple = tuple(Ing_lst) if len(Ing_lst) > 1 else f"('{Ing_lst[0]}')"

    query = f"""
    SELECT r2.RXCUI as RXCUI, r2.STR as STR
    FROM RXNREL r1
    JOIN RXNCONSO r2 ON r1.RXCUI2 = r2.RXCUI
    WHERE r1.RXCUI1 IN {ing_tuple} 
      AND r2.TTY = 'DP'
      AND r2.RXCUI != :current_id
    GROUP BY r2.RXCUI, r2.STR
    HAVING COUNT(DISTINCT r1.RXCUI1) = :ing_count
    """
    
    with engine.connect() as conn:
        res = pd.read_sql(text(query), conn, params={
            'current_id': current_id, 
            'ing_count': len(Ing_lst)
        })
    
    if not res.empty:
        res = res[res['STR'].str.contains(r'\[.*\]', na=False)].copy()
        res = extract_name(res)
        if ing_names:
            for name in ing_names:
                res = res[~res['Product_Name'].str.contains(name, case=False, na=False)]
        res.reset_index(drop=True, inplace=True)
    return res

def Fetch_Dose_Form(ID):
    query = text("SELECT c.STR FROM RXNCONSO c JOIN RXNREL r ON c.RXCUI = r.RXCUI2 WHERE r.RXCUI1 = :id AND c.TTY = 'DF'")
    with engine.connect() as conn:
        res = pd.read_sql(query, conn, params={'id': ID})
    return res["STR"].iloc[0] if not res.empty else "Not specified"

def fetch_generic_name(ID):
    query = text("SELECT c.STR FROM RXNCONSO c JOIN RXNREL r ON c.RXCUI = r.RXCUI2 WHERE r.RXCUI1 = :id AND c.TTY = 'SCD'")
    with engine.connect() as conn:
        res = pd.read_sql(query, conn, params={'id': ID})
    return res["STR"].iloc[0] if not res.empty else "N/A"

def Fetch_Heatmap(df, drug_of_interest_id, drug_of_interest_name):
    searched_row = pd.DataFrame({
        "ID": [drug_of_interest_id],
        "Product_Name": [drug_of_interest_name]
    })
    df_extended = pd.concat([searched_row, df], ignore_index=True)
    
    rows = []
    for _, row in df_extended.iterrows():
        ingredients = Fetch_Ingredients(row["ID"])
        for _, ing in ingredients.iterrows():
            rows.append({
                "ID": row["ID"], 
                "Product_Name": row["Product_Name"],
                "Ingredient": ing["Ingredient"],
                "Concentration": ing["Concentration"]
            })

    long_df = pd.DataFrame(rows)
    if long_df.empty:
        return pd.DataFrame()

    heatmap_df = long_df.pivot_table(
        index=["ID", "Product_Name"], 
        columns="Ingredient",
        values="Concentration",
        fill_value=0
    ).reset_index()

    return heatmap_df

def Create_Altair_Heatmap(heatmap_df, drug_of_interest_id):
    if heatmap_df.empty:
        return alt.Chart(pd.DataFrame({'text': ['No data to display']})).mark_text().encode(text='text:N')

    cols_to_norm = heatmap_df.columns.difference(['ID', 'Product_Name'])
    norm_df = heatmap_df.copy()
    
    norm_df[cols_to_norm] = norm_df[cols_to_norm].apply(
        lambda x: x / x.max() if x.max() != 0 else 0
    )

    df_long = norm_df.melt(
        id_vars=["ID", "Product_Name"],
        var_name="Ingredient",
        value_name="Relative_Conc"
    )

    raw_long = heatmap_df.melt(
        id_vars=["ID", "Product_Name"],
        var_name="Ingredient",
        value_name="Raw_Concentration"
    )
    df_long["Concentration"] = raw_long["Raw_Concentration"]

    df_long = df_long[df_long["Relative_Conc"] > 0].copy()
    df_long["Is_Interest"] = df_long["ID"].astype(str) == str(drug_of_interest_id)

    chart = alt.Chart(df_long).mark_rect().encode(
        x=alt.X('Ingredient:N', axis=alt.Axis(labelAngle=-45)),
        y=alt.Y('Product_Name:N', sort=None),
        color=alt.Color(
            'Relative_Conc:Q',
            scale=alt.Scale(scheme='blues', domain=[0, 1]),
            title='Relative Conc.'
        ),
        stroke=alt.condition(
            alt.datum.Is_Interest, 
            alt.value('black'), 
            alt.value(None)
        ),
        strokeWidth=alt.condition(
            alt.datum.Is_Interest, 
            alt.value(2.5), 
            alt.value(0)
        ),
        tooltip=[
            'Product_Name',
            'Ingredient',
            alt.Tooltip('Concentration:Q', title='Actual Dose (mg)'),
            alt.Tooltip('Relative_Conc:Q', format='.2f', title='Rel. Strength')
        ]
    ).properties(
        width='container',
        height=400,
        title="Normalized Ingredient Heatmap (Related Drugs)"
    )

    return chart

In [13]:
ingredients = Fetch_Ingredients(209387)
Fetch_Related_Drugs(ingredients["Ingredient_ID"].tolist(), 209387)


,RXCUI,STR,Product_Name
0,1042688,ACETAMINOPHEN 325 mg / PHENYLEPHRINE HYDROCHLO...,Cough And Cold Plus
1,1046378,ACETAMINOPHEN 325 mg / PHENYLEPHRINE HYDROCHLO...,Coltalin-Nd
2,1046751,ACETAMINOPHEN 325 mg / DIPHENHYDRAMINE HYDROCH...,Allergy Relief
3,1046781,ACETAMINOPHEN 325 mg / CHLORPHENIRAMINE MALEAT...,Cold Medicine Xl3
4,1049214,OXYCODONE HYDROCHLORIDE 10 mg / ACETAMINOPHEN ...,Oxycodone And Acetaminophen
5,1049216,OXYCODONE HYDROCHLORIDE 10 mg / ACETAMINOPHEN ...,Endocet
6,1049625,OXYCODONE HYDROCHLORIDE 10 mg / ACETAMINOPHEN ...,Percocet
7,1052466,ACETAMINOPHEN 325 mg / DIPHENHYDRAMINE HYDROCH...,Percogesic Original Strength
8,1052647,ACETAMINOPHEN 325 mg / DOXYLAMINE SUCCINATE 6....,Nighttime Sinus Relief
9,1052670,ACETAMINOPHEN 325 mg / PHENYLEPHRINE HYDROCHLO...,Daytime Sinus Relief


In [10]:
Fetch_Related_Drugs(209387)

TypeError: Fetch_Related_Drugs() missing 1 required positional argument: 'current_id'

NameError: name 'ingredients' is not defined